# Comparison of *in situ* observsations and the ERA5 reanalysis climatology

[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/ecmwf-training/c3s-training-submodule-insitu-obs/main?labpath=insitu-obs-against-climatology.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/c3s-training-submodule-insitu-obs/blob/main/insitu-obs-against-climatology.ipynb)
[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/c3s-training-submodule-insitu-obs/blob/main/insitu-obs-against-climatology.ipynb)

:::{note}
This notebook can be run on free online platforms, such as Binder, Kaggle and Colab, or they can be accessed from GitHub. The links to run this notebook in these environments are provided here, but please note they are not supported by ECMWF.
:::

*Provide an introduction and context for the notebook here.*


## Learning objectives 🎯


This notebook will teach you how to:
1. Access data from the Climate Data Store using `earthkit-data`
2. Calculate climatological averages using `earthkit-transforms`
3. Create publication quality figures comparing the data sources using `earthkit-plots`


## Prepare your environment


*Insert here any necessary instructions to set-up the environment of learners, any background information on data, projects, access to data catalogues, or preliminary steps necessary to run the notebooks. These may include instructions for installing packages, imports, etc.*

:::{important}
All required dependencies should be included in the `environment.yml` file.
:::

### Import libraries


In [1]:
# Import your libraries here
import earthkit.data as ekd
import earthkit.transforms as ekt
import earthkit.plots as ekp

import numpy as np
import pandas as pd

### Select a storm of interest

For this example demonstration, we will look at the
[Great Storm of 1987](https://en.wikipedia.org/wiki/Great_storm_of_1987).
The storm occured on the night of the 15th/16th October, we will look at the
antecedent build up of meteorological conditions and compare the observations
with the long-term climatological behaviour.

The [Windstorm tracks and footprints over Europe](https://cds.climate.copernicus.eu/datasets/sis-european-wind-storm-reanalysis?tab=overview)
dataset on the CDS has a number of wind storms to explore.

In [2]:
# Period and region of interest
year = "1987"
month = "10"
day = "15"

In [3]:
storm_dataset = "sis-european-wind-storm-reanalysis"
storm_request = {
    "product": "windstorm_track",
    "variable": "all",
    "tracking_algorithm": ["hodges"],
    "event_aggregation": "single_event",
    "year": year,
    "month": month,
    "day": day
}
storm_ekds = ekd.from_source("cds", storm_dataset, storm_request)
storm_df = storm_ekds.to_pandas()
storm_df["time"] = pd.to_datetime(storm_df["time"])
storm_df

,id,time,latitude,longitude,fg10,lsm,msl,algorithm
0,1014,1987-10-15 00:00:00,42.25,-23.25,12.154546,0.000000,98603.690,hodges
1,1014,1987-10-15 06:00:00,42.50,-17.50,5.343058,0.000000,97911.625,hodges
2,1014,1987-10-15 12:00:00,44.25,-13.50,16.133911,0.000000,96682.875,hodges
3,1014,1987-10-15 18:00:00,46.50,-9.00,16.743628,0.000000,95850.000,hodges
4,1014,1987-10-16 00:00:00,49.25,-4.50,24.250221,0.000000,95340.810,hodges
5,1014,1987-10-16 06:00:00,53.50,-0.75,14.803338,0.997857,95791.375,hodges
6,1014,1987-10-16 12:00:00,58.50,0.00,13.904638,0.000000,95646.750,hodges
7,1014,1987-10-16 18:00:00,61.50,0.50,8.246316,0.000000,95929.500,hodges
8,1014,1987-10-17 00:00:00,63.25,0.50,3.650280,0.000000,96462.810,hodges
9,1014,1987-10-17 06:00:00,66.25,1.50,12.069446,0.000000,96664.690,hodges


With our storm selected, we can define an area and date range based on the information in the storm track DataFrame. We add some padding to make sure we capture enough of the weather system

In [39]:
min_lat = np.floor(storm_df.latitude.min() - 2).item()
max_lat = np.ceil(storm_df.latitude.max() + 2).item()
min_lon = np.floor(storm_df.longitude.min() - 2).item()
max_lon = np.ceil(storm_df.longitude.max() + 2).item()
area = [max_lat, min_lon, min_lat, max_lon]

start_date = storm_df.time.min()-pd.Timedelta(days=3)
end_date = storm_df.time.max()+pd.Timedelta(days=1)
daterange = f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}"
daterange

'1987-10-12/1987-10-18'

## Download data from the CDS

Earthkit-data uses your CDS-API credentials to download data, it is advised that you set up
a `~/.cdsapirc` file with your credentials following the
[how to api instructions](https://cds.climate.copernicus.eu/how-to-api).
If you have not setup your `~/.cdsapirc` file, you will be prompted for your credentials
when executing the cells below.

### ERA5 Hourly data

In [61]:
era5_hourly_dataset = "reanalysis-era5-single-levels"
variables = [
    "2m_temperature",
    "total_precipitation",
    "sea_surface_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_dewpoint_temperature",
    "mean_sea_level_pressure",
    # "mean_wave_direction",
    # "mean_wave_period",
    # "significant_height_of_combined_wind_waves_and_swell",
    "surface_pressure",
]
era5_hourly_request = {
    "product_type": ["reanalysis"],
    "variable": variables,
    "date": daterange,
    "time": [f"{i:02d}:00" for i in range(24)],
    "area": area,
}

era5_hourly_ekds = ekd.from_source("cds", era5_hourly_dataset, era5_hourly_request)
era5_hourly_ds = era5_hourly_ekds.to_xarray(time_dim_mode="valid_time")
era5_hourly_ds

2026-04-15 17:50:17,137 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-15 17:50:17,139 INFO Request ID is 7543c6aa-d213-4fec-b073-2351ffefd281
2026-04-15 17:50:17,458 INFO status has been updated to accepted
2026-04-15 17:50:29,090 INFO status has been updated to running
2026-04-15 17:53:14,969 INFO status has been updated to successful


e21c8246ad1945127c1742bc476eae28.grib:   0%|          | 0.00/46.0M [00:00<?, ?B/s]

<xarray.Dataset> Size: 196MB
Dimensions:     (valid_time: 168, latitude: 129, longitude: 141)
Coordinates:
  * valid_time  (valid_time) datetime64[us] 1kB 1987-10-12 ... 1987-10-18T23:...
  * latitude    (latitude) float64 1kB 72.0 71.75 71.5 71.25 ... 40.5 40.25 40.0
  * longitude   (longitude) float64 1kB -26.0 -25.75 -25.5 ... 8.5 8.75 9.0
Data variables:
    10u         (valid_time, latitude, longitude) float64 24MB ...
    10v         (valid_time, latitude, longitude) float64 24MB ...
    2d          (valid_time, latitude, longitude) float64 24MB ...
    2t          (valid_time, latitude, longitude) float64 24MB ...
    msl         (valid_time, latitude, longitude) float64 24MB ...
    sp          (valid_time, latitude, longitude) float64 24MB ...
    sst         (valid_time, latitude, longitude) float64 24MB ...
    tp          (valid_time, latitude, longitude) float64 24MB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

# Plot maps of the storm progression

In [62]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

if "valid_time" not in era5_hourly_ds.dims:
    raise ValueError("`era5_hourly_ds` must contain a `valid_time` dimension.")

valid_time_vars = [
    name for name in era5_hourly_ds.data_vars
    if "valid_time" in era5_hourly_ds[name].dims
]
if not valid_time_vars:
    raise ValueError("No data variables with a `valid_time` dimension were found.")

lat_name = "latitude" if "latitude" in era5_hourly_ds.coords else "lat"
lon_name = "longitude" if "longitude" in era5_hourly_ds.coords else "lon"
if lat_name not in era5_hourly_ds.coords or lon_name not in era5_hourly_ds.coords:
    raise ValueError("Expected latitude/longitude coordinates in `era5_hourly_ds`.")

valid_times = pd.to_datetime(era5_hourly_ds["valid_time"].values)

variable_dropdown = widgets.Dropdown(
    options=valid_time_vars,
    value=valid_time_vars[0],
    description="Variable:",
    layout=widgets.Layout(width="60%"),
    style={"description_width": "initial"},
)

time_slider = widgets.SelectionSlider(
    options=[(t.strftime("%Y-%m-%d %H:%M"), i) for i, t in enumerate(valid_times)],
    value=0,
    description="valid_time:",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
    style={"description_width": "initial"},
)

plot_output = widgets.Output()

def update_map(variable, time_index):
    da = era5_hourly_ds[variable].isel(valid_time=time_index).squeeze()
    fig = ekp.quickplot(da)

    with plot_output:
        clear_output(wait=True)
        display(fig.fig)
        plt.close(fig.fig)

def on_control_change(_change):
    update_map(variable_dropdown.value, time_slider.value)

variable_dropdown.observe(on_control_change, names="value")
time_slider.observe(on_control_change, names="value")

display(widgets.VBox([variable_dropdown, time_slider, plot_output]))
update_map(variable_dropdown.value, time_slider.value)

### *in situ* data

Select the 

In [45]:
start_day = start_date.day
end_day = end_date.day
if start_date.month != end_date.month:
    raise ValueError("Start and end dates must be within the same month for this example.")
dataset = "insitu-observations-surface-marine"
request = {
    "version": "2_0_0",
    "variable": [
        "air_pressure_at_sea_level",
        "air_temperature",
        "dew_point_temperature",
        "water_temperature",
        "wind_from_direction",
        "wind_speed"
    ],
    "year": start_date.year,
    "month": start_date.month,
    "day": [i for i in range(start_day, end_day + 1)],
    "area": area,
}
insitu_ekds = ekd.from_source("cds", dataset, request)
insitu_ds = insitu_ekds.to_xarray()
insitu_ds["observed_variable"] = insitu_ds.observed_variable.str.decode("UTF-8")
# Assign some of the data variables as coordinates for easier selection and plotting
for coord in ["latitude", "longitude", "report_timestamp", "observed_variable", "observation_height_above_station_surface"]:
    insitu_ds = insitu_ds.assign_coords({coord: insitu_ds[coord]})
unique_observed_variables = insitu_ds.observed_variable.to_index().unique()

insitu_ds

2026-04-15 17:17:20,006 INFO Request ID is 515432af-8ddd-4f31-850e-e8e71c8bf48a
2026-04-15 17:17:20,083 INFO status has been updated to accepted
2026-04-15 17:17:41,375 INFO status has been updated to successful


b3c6df6886005259bd0d96e29f5c5093.nc:   0%|          | 0.00/447k [00:00<?, ?B/s]

<xarray.Dataset> Size: 8MB
Dimensions:                                   (index: 33575)
Coordinates:
    longitude                                 (index) float32 134kB ...
    latitude                                  (index) float32 134kB ...
    observation_height_above_station_surface  (index) float32 134kB ...
    observed_variable                         (index) <U25 3MB 'air_temperatu...
    report_timestamp                          (index) datetime64[ns] 269kB ...
Dimensions without coordinates: index
Data variables: (12/15)
    observation_id                            (index) |S26 873kB ...
    report_id                                 (index) |S22 739kB ...
    data_policy_licence                       (index) int32 134kB ...
    observation_value                         (index) float64 269kB ...
    value_significance                        (index) int32 134kB ...
    units                                     (index) |S3 101kB ...
    ...                                        ...
    station_name                              (index) |S25 839kB ...
    platform_type                             (index) int32 134kB ...
    primary_station_id                        (index) |S9 302kB ...
    height_of_station_above_sea_level         (index) float32 134kB ...
    report_meaning_of_timestamp               (index) int32 134kB ...
    report_duration                           (index) int32 134kB ...
Attributes:
    featureType:               point
    contactemail:              https://support.ecmwf.int
    licence_list:              licence-to-use-copernicus-products
    responsible_organisation:  ECMWF

In [4]:
era5_climatologies = ekt.climatology.monthly_mean(era5_ds)
era5_climatologies

<xarray.Dataset> Size: 4MB
Dimensions:    (month: 12, latitude: 81, longitude: 161)
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float64 648B 65.0 64.75 64.5 64.25 ... 45.5 45.25 45.0
  * longitude  (longitude) float64 1kB -10.0 -9.75 -9.5 ... 29.5 29.75 30.0
Data variables:
    2t         (month, latitude, longitude) float64 1MB 274.9 274.9 ... 278.5
    tp         (month, latitude, longitude) float64 1MB 0.002677 ... 0.00177
    sst        (month, latitude, longitude) float64 1MB 276.3 276.2 ... 281.8
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

Select the hourly data from ERA5 single levels.

In [5]:
era5_hourly_dataset = "reanalysis-era5-single-levels"
variables = [
    "2m_temperature",
    "total_precipitation",
    "sea_surface_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_dewpoint_temperature",
    "mean_sea_level_pressure",
    "mean_wave_direction",
    "mean_wave_period",
    "significant_height_of_combined_wind_waves_and_swell",
    "surface_pressure",
]
era5_hourly_request = {
    "product_type": ["reanalysis"],
    "variable": variables,
    "year": year,
    "month": month,
    "day": days,
    "time": ["00:00"],
    "area": area,
}

era5_hourly_ekds = ekd.from_source("cds", era5_hourly_dataset, era5_hourly_request)
era5_hourly_ds = era5_hourly_ekds.to_xarray(time_dim_mode="valid_time")
era5_hourly_ds

<xarray.Dataset> Size: 5MB
Dimensions:     (valid_time: 4, latitude: 81, longitude: 161)
Coordinates:
  * valid_time  (valid_time) datetime64[us] 32B 1987-10-14 ... 1987-10-17
  * latitude    (latitude) float64 648B 65.0 64.75 64.5 ... 45.5 45.25 45.0
  * longitude   (longitude) float64 1kB -10.0 -9.75 -9.5 ... 29.5 29.75 30.0
Data variables:
    10u         (valid_time, latitude, longitude) float64 417kB ...
    10v         (valid_time, latitude, longitude) float64 417kB ...
    2d          (valid_time, latitude, longitude) float64 417kB ...
    2t          (valid_time, latitude, longitude) float64 417kB ...
    msl         (valid_time, latitude, longitude) float64 417kB ...
    mwd         (valid_time, latitude, longitude) float64 417kB ...
    mwp         (valid_time, latitude, longitude) float64 417kB ...
    sp          (valid_time, latitude, longitude) float64 417kB ...
    sst         (valid_time, latitude, longitude) float64 417kB ...
    swh         (valid_time, latitude, longitude) float64 417kB ...
    tp          (valid_time, latitude, longitude) float64 417kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

In [8]:
unique_observed_variables

Index(['air_temperature', 'water_temperature', 'wind_from_direction',
       'wind_speed', 'air_pressure_at_sea_level', 'dew_point_temperature'],
      dtype='str', name='index')

In [3]:
era5_dataset = "reanalysis-era5-single-levels-monthly-means"
variables = [
    "2m_temperature",
    "total_precipitation",
    "sea_surface_temperature",
    # "10m_u_component_of_wind",
    # "10m_v_component_of_wind",
    # "2m_dewpoint_temperature",
    # "mean_sea_level_pressure",
    # "mean_wave_direction",
    # "mean_wave_period",
    # "significant_height_of_combined_wind_waves_and_swell",
    # "surface_pressure",
]
base_request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "year": [f"{year:04d}" for year in range(1961, 1990)],
    "month": [f"{month:02d}" for month in range(1, 13)],
    "time": ["00:00"],
    "area": area,
}

era5_datasets = {}
for var in variables:
    request = base_request.copy()
    request["variable"] = var
    era5_ekds = ekd.from_source("cds", era5_dataset, request)
    era5_datasets[var] = era5_ekds.to_xarray(time_dim_mode="valid_time")

# Update the Accumulated variables to use the same dimension as the other variables for easier comparison
ref_var = "2m_temperature"
for var in ["total_precipitation"]:
    era5_datasets[var] = era5_datasets[var].assign_coords({"valid_time": era5_datasets[ref_var].valid_time})

# Now we can merge the datasets into a single xarray Dataset
import xarray as xr
era5_ds = xr.merge(era5_datasets.values())
era5_ds

<xarray.Dataset> Size: 109MB
Dimensions:     (valid_time: 348, latitude: 81, longitude: 161)
Coordinates:
  * valid_time  (valid_time) datetime64[us] 3kB 1961-01-01 ... 1989-12-01
  * latitude    (latitude) float64 648B 65.0 64.75 64.5 ... 45.5 45.25 45.0
  * longitude   (longitude) float64 1kB -10.0 -9.75 -9.5 ... 29.5 29.75 30.0
Data variables:
    2t          (valid_time, latitude, longitude) float64 36MB ...
    tp          (valid_time, latitude, longitude) float64 36MB ...
    sst         (valid_time, latitude, longitude) float64 36MB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

### Storm Track data

The CDS also offers Storm Track data which can be used to map storms

## Take home messages 📌


*In this section, summarise key take home messages.*

- *Key message 1*
- *Key message 2*
- *Key message 3*
